# Somnotate Testing
Run Somnotate on annotated .mat files in data/hypnose_eeg/somnotate_testing,
then visualize scores against manual annotations and compute agreements.

In [11]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from utils.testing import (
    ensure_csvs,
    ensure_somnotate_predictions,
    load_aligned_vectors,
    get_csv_dir,
    get_predictions_dir,
    get_test_root,
    plot_testing_comparison,
    plot_testing_comparison_detailed,
    agreement_matrix,
    plot_agreement_matrix,
    print_agreement_diagnostic,
)
%matplotlib qt


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
model_name = "test_model_2"
current_test = "somnotate_performance_test"
repo_root = Path.cwd().parent

testing_mat_dir = repo_root / "data" / "hypnose_eeg" / "somnotate_testing" / current_test
model_path = (
    repo_root
    / "data" / "hypnose_eeg" / "derivatives" / "somnotate_training"
    / model_name / "model.pickle"
)

# Derived paths used by every cell below — kept here so each cell can run independently.
test_root = get_test_root(repo_root, model_name, current_test)
csv_dir = get_csv_dir(repo_root, model_name, current_test)
predictions_dir = get_predictions_dir(repo_root, model_name, current_test)
agreement_plot_path = test_root / "agreement_matrix.png"

print("testing_mat_dir:", testing_mat_dir)
print("model_path:    ", model_path)
print("csv_dir:       ", csv_dir)
print("predictions_dir:", predictions_dir)


testing_mat_dir: /Users/joschua/repos/harris_lab/hypnose/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/somnotate_performance_test
model_path:     /Users/joschua/repos/harris_lab/hypnose/eeg_preprocessing/data/hypnose_eeg/derivatives/somnotate_training/test_model_2/model.pickle
csv_dir:        /Volumes/harris/hypnose/hypnose_eeg/derivatives/somnotate_testing/test_model_2/somnotate_performance_test/intermediate/csv
predictions_dir: /Volumes/harris/hypnose/hypnose_eeg/derivatives/somnotate_testing/test_model_2/somnotate_performance_test/somnotate_predictions


In [3]:
# Run once to materialise CSVs + somnotate predictions.
# Skips work if outputs already exist; pass force=True to regenerate.
csv_dir, csv_files = ensure_csvs(
    testing_mat_dir=testing_mat_dir,
    repo_root=repo_root,
    model_name=model_name,
    test_name=current_test,
)
print(f"CSV files: {len(csv_files)}")
for path in csv_files:
    print(" -", path.name)

somnotate_pred_path = ensure_somnotate_predictions(
    csv_files=csv_files,
    predictions_dir=predictions_dir,
    model_path=model_path,
)
print("Somnotate predictions:", somnotate_pred_path)


Using 4 existing CSV(s) in /Volumes/harris/hypnose/hypnose_eeg/derivatives/somnotate_testing/test_model_2/somnotate_performance_test/intermediate/csv
CSV files: 4
 - sub-053_ses-017-recording-001_CONSENSUS.csv
 - sub-053_ses-017_recording-001_Joschua.csv
 - sub-053_ses-017_recording-001_Nam.csv
 - sub-053_ses-017_recording-001_Volkan.csv
Using existing predictions: /Volumes/harris/hypnose/hypnose_eeg/derivatives/somnotate_testing/test_model_2/somnotate_performance_test/somnotate_predictions/sub-006_ses-01_recording-01_Nam_somnotate.parquet
Somnotate predictions: /Volumes/harris/hypnose/hypnose_eeg/derivatives/somnotate_testing/test_model_2/somnotate_performance_test/somnotate_predictions/sub-006_ses-01_recording-01_Nam_somnotate.parquet


In [4]:
# Visualizer — loads vectors from disk if not already in memory.
try:
    raw_signals, somnotate_vec, manual_vectors
except NameError:
    raw_signals, somnotate_vec, manual_vectors = load_aligned_vectors(csv_dir, predictions_dir)

fig, viewer = plot_testing_comparison(
    raw_signals,
    sampling_rate_hz=512,
    somnotate_vec=somnotate_vec,
    manual_vectors=manual_vectors,
)


KeyboardInterrupt: 

In [15]:
# Detailed visualizer — pick which scorers to show + extra derived channels.
# scorers=None → all scorers. Set to e.g. ["CONSENSUS"] to compare somnotate vs the consensus only.
try:
    raw_signals, somnotate_vec, manual_vectors
except NameError:
    raw_signals, somnotate_vec, manual_vectors = load_aligned_vectors(csv_dir, predictions_dir)

fig_detailed, viewer_detailed = plot_testing_comparison_detailed(
    raw_signals,
    sampling_rate_hz=512,
    somnotate_vec=somnotate_vec,
    manual_vectors=manual_vectors,
    scorers=["CONSENSUS"],   # e.g. ["Joschua", "Nam"] or None for all
    eeg_channel=0,             # 0 → EEG1, 1 → EEG2
    view_length_s=120.0,
    band_smoothing_window_s=5.0,
)


Computing derived signal channels…


In [5]:
# Agreement matrix — loads vectors from disk if not already in memory.
try:
    somnotate_vec, manual_vectors
except NameError:
    _, somnotate_vec, manual_vectors = load_aligned_vectors(csv_dir, predictions_dir)

agreement_df = agreement_matrix(somnotate_vec, manual_vectors)
plot_agreement_matrix(agreement_df, output_path=agreement_plot_path)
print("Saved agreement matrix:", agreement_plot_path)
agreement_df


OSError: [Errno 9] Bad file descriptor

In [ ]:
# Diagnostic — loads vectors from disk if not already in memory.
# Per-vector code histograms + per-state confusion matrix for each scorer vs somnotate.
try:
    somnotate_vec, manual_vectors
except NameError:
    _, somnotate_vec, manual_vectors = load_aligned_vectors(csv_dir, predictions_dir)

print_agreement_diagnostic(somnotate_vec, manual_vectors)
